# 02. Sistema de Recomendación Basado en Contenido (Sinopsis / NLP)

## Introducción
En lugar de sugerir únicamente las películas más populares en general, ahora construiremos un sistema que recomiende películas similares a una película específica que le guste al usuario.

Para lograr esto utilizaremos técnicas de **Procesamiento de Lenguaje Natural (NLP)** sobre las descripciones de la trama (columna `overview`).
1. Transformaremos el texto de las sinopsis en vectores numéricos usando **TF-IDF** (*Term Frequency - Inverse Document Frequency*).
2. Calcularemos la similitud entre cada par de películas mediante la **Similitud Coseno** usando `linear_kernel`.
3. Crearemos una función que, dado el título de una película, devuelva las 10 más similares.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Cargar dataset de películas
df = pd.read_csv('./data/movies_metadata.csv', low_memory=False)
df.head(3)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


## 1. Exploración de las Sinopsis (`overview`)

In [2]:
# Inspeccionar las primeras descripciones
df['overview'].head()

0    Led by Woody, Andy's toys live happily in his ...
1    When siblings Judy and Peter discover an encha...
2    A family wedding reignites the ancient feud be...
3    Cheated on, mistreated and stepped on, the wom...
4    Just when George Banks has recovered from his ...
Name: overview, dtype: str

## 2. Vectorización TF-IDF
La puntuación TF-IDF cuantifica la frecuencia de una palabra en una sinopsis particular, ponderándola negativamente por la frecuencia con la que aparece en todo el catálogo de películas. Esto ayuda a restarle peso a palabras muy comunes y destacar los términos temáticos distintivos.

Pasos a realizar:
* Eliminar palabras vacías (*stop words*) en inglés (ej. *the, a, is, in*).
* Reemplazar valores nulos (`NaN`) por cadenas vacías.
* Construir la matriz TF-IDF.

In [3]:
# Definir el objeto TF-IDF Vectorizer eliminando stop words en inglés
tfidf = TfidfVectorizer(stop_words='english')

# Reemplazar valores nulos por cadenas vacías
df['overview'] = df['overview'].fillna('')

# Construir la matriz TF-IDF
tfidf_matrix = tfidf.fit_transform(df['overview'])

print("Dimensiones de la matriz TF-IDF:", tfidf_matrix.shape)

Dimensiones de la matriz TF-IDF: (45466, 75827)


In [4]:
# Ver algunas de las características (palabras) aprendidas
vocabulario = tfidf.get_feature_names_out()
print("Muestra del vocabulario (índices 5000 a 5010):", vocabulario[5000:5010])

Muestra del vocabulario (índices 5000 a 5010): ['avails' 'avaks' 'avalanche' 'avalanches' 'avallone' 'avalon' 'avant'
 'avanthika' 'avanti' 'avaracious']


## 3. Cálculo de la Similitud Coseno

La similitud coseno entre dos vectores $x$ e $y$ se define como:

$$\cos(x,y) = \frac{\bf x \cdot \bf y^{\intercal}}{\lVert \bf x \rVert \cdot \lVert \bf y \rVert}$$

Como los vectores calculados por `TfidfVectorizer` ya están normalizados en norma $L2$, calcular el producto punto lineal directo entre ellos (`linear_kernel`) equivale a la similitud coseno y es computacionalmente más rápido que `cosine_similarity`.

In [5]:
# Calcular la matriz de similitud coseno completa
# NOTA: La matriz resultante es de 45466 x 45466 elementos.
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

print("Dimensiones de la matriz de similitud:", cosine_sim.shape)

Dimensiones de la matriz de similitud: (45466, 45466)


## 4. Construcción del Mapeo Inverso de Títulos e Índices
Necesitamos poder consultar rápidamente la posición (índice) de una película a partir de su título.

In [6]:
# Mapeo inverso de títulos a índices del DataFrame (eliminando duplicados de títulos)
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
indices[:10]

title
Toy Story                      0
Jumanji                        1
Grumpier Old Men               2
Waiting to Exhale              3
Father of the Bride Part II    4
Heat                           5
Sabrina                        6
Tom and Huck                   7
Sudden Death                   8
GoldenEye                      9
dtype: int64

## 5. Función de Recomendación
La función `get_recommendations` realiza los siguientes pasos:
1. Localiza el índice de la película por su título.
2. Extrae las puntuaciones de similitud coseno con todas las demás películas.
3. Ordena los resultados en forma descendente.
4. Toma las 10 mejores recomendaciones (omitiendo la posición 0, que es la misma película solicitada).
5. Retorna los títulos recomendados.

In [7]:
def get_recommendations(title, cosine_sim=cosine_sim, df=df, indices=indices):
    # Validar existencia del título
    if title not in indices:
        return f"Película '{title}' no encontrada en el catálogo."
        
    # Obtener el índice de la película
    idx = indices[title]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    # Obtener las puntuaciones de similitud de todas las películas con esa película
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Ordenar las películas según la puntuación de similitud
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Tomar las 10 películas más similares (ignorando la primera que es ella misma)
    sim_scores = sim_scores[1:11]

    # Obtener los índices de las películas recomendadas
    movie_indices = [i[0] for i in sim_scores]

    # Devolver los títulos correspondientes
    return df['title'].iloc[movie_indices]

## 6. Pruebas del Modelo

In [8]:
# Prueba 1: The Dark Knight Rises
get_recommendations('The Dark Knight Rises')

12481                                      The Dark Knight
150                                         Batman Forever
1328                                        Batman Returns
15511                           Batman: Under the Red Hood
585                                                 Batman
21194    Batman Unmasked: The Psychology of the Dark Kn...
9230                    Batman Beyond: Return of the Joker
18035                                     Batman: Year One
19792              Batman: The Dark Knight Returns, Part 1
3095                          Batman: Mask of the Phantasm
Name: title, dtype: str

In [9]:
# Prueba 2: The Godfather
get_recommendations('The Godfather')

1178               The Godfather: Part II
44030    The Godfather Trilogy: 1972-1990
1914              The Godfather: Part III
23126                          Blood Ties
11297                    Household Saints
34717                   Start Liquidation
10821                            Election
38030            A Mother Should Be Loved
17729                   Short Sharp Shock
26293                  Beck 28 - Familjen
Name: title, dtype: str

## Análisis de Resultados y Limitaciones
* Observamos que al consultar *The Dark Knight Rises*, el modelo devuelve casi exclusivamente películas de Batman porque comparten palabras clave en la sinopsis ("Batman", "Gotham", "vigilante").
* Sin embargo, alguien que disfrutó *The Dark Knight Rises* probablemente también apreciaría otras obras de **Christopher Nolan** (*Inception*, *The Prestige*, *Interstellar*), algo que este recomendador basado solo en texto no puede inferir.
* En el cuaderno **`03_recomendador_contenido_metadatos.ipynb`** enriqueceremos este modelo utilizando metadatos detallados: **director, actores principales, palabras clave temáticas y géneros**.